In [0]:
-- =====================================================================
-- CELL 1: AUDIT -> who accessed our project's data, and how (last 7 days)
-- =====================================================================
-- system.access.audit records every Unity Catalog action.
-- request_params is a MAP (key -> value); ['full_name_arg'] reads one key.
CREATE OR REPLACE VIEW procintel.ops.v_audit_access AS
SELECT event_time,
       user_identity.email               AS who,
       action_name                       AS action,
       request_params['full_name_arg']   AS object,
       response.status_code              AS status
FROM system.access.audit
WHERE service_name = 'unityCatalog'
  AND request_params['full_name_arg'] LIKE 'procintel.%'
  AND event_date >= current_date() - INTERVAL 7 DAYS;

SELECT who, action, COUNT(*) AS n
FROM procintel.ops.v_audit_access
GROUP BY ALL ORDER BY n DESC LIMIT 20;

In [0]:
-- =====================================================================
-- CELL 2: LINEAGE -> which table was built from which (automatic in Unity Catalog)
-- =====================================================================
-- Databricks records lineage for every query: no extra code needed.
-- This proves the medallion flow: bronze -> silver -> gold -> agents.
CREATE OR REPLACE VIEW procintel.ops.v_lineage AS
SELECT DISTINCT source_table_full_name AS source, target_table_full_name AS target
FROM system.access.table_lineage
WHERE target_table_full_name LIKE 'procintel.%'
  AND source_table_full_name IS NOT NULL;

SELECT * FROM procintel.ops.v_lineage ORDER BY target;

In [0]:
-- =====================================================================
-- CELL 3: SYSTEM LOG -> SQL activity on the warehouse (dashboard + Data agent)
-- =====================================================================
CREATE OR REPLACE VIEW procintel.ops.v_query_activity AS
SELECT date(start_time)             AS day,
       executed_by                  AS who,
       execution_status             AS status,
       COUNT(*)                     AS queries,
       ROUND(AVG(total_duration_ms)) AS avg_ms
FROM system.query.history
WHERE start_time >= current_date() - INTERVAL 7 DAYS
GROUP BY ALL;

SELECT * FROM procintel.ops.v_query_activity ORDER BY day DESC, queries DESC;

In [0]:
-- =====================================================================
-- CELL 4: APPLICATION LOG summary -> how is the chatbot behaving?
-- =====================================================================
SELECT COUNT(*)                                         AS requests,
       ROUND(100 * AVG(CASE WHEN verdict = 'PASS' THEN 1 ELSE 0 END), 1) AS pct_verified,
       ROUND(percentile(latency_ms, 0.5) / 1000, 1)     AS median_latency_s,
       ROUND(percentile(latency_ms, 0.95) / 1000, 1)    AS p95_latency_s
FROM procintel.ops.agent_requests;